In [8]:
import os
import json
from pathlib import Path

import random
import pandas as pd
import numpy as np
import torch
import seaborn as sns
import matplotlib.pyplot as plt
import cv2
from tqdm import tqdm
import ast
import re


In [7]:
2+2

4

# Verify vjepa backbone architecture
####  apiantonio/vjepa2.1-vit-base-384

In [4]:
import torch
from transformers import AutoModel, AutoConfig, AutoVideoProcessor

MODEL_ID = "apiantonio/vjepa2.1-vit-base-384"
processor = AutoVideoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

I0000 00:00:1791965782.248253 1879910 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [9]:
import torch
from transformers import AutoModel, AutoConfig

MODEL_ID = "apiantonio/vjepa2.1-vit-base-384"

# 1. Load config and model directly using local Hugging Face cache
# (Since trust_remote_code=True is required for custom model architectures like V-JEPA 2.1)
config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)
model.eval()

I0000 00:00:1791980230.264925 2440096 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


VJEPA21Model(
  (encoder): VJEPA21Encoder(
    (embeddings): VJEPA21Embeddings(
      (patch_embeddings): VJEPA21PatchEmbeddings3D(
        (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (patch_embeddings_img): VJEPA21PatchEmbeddings3D(
        (proj): Conv3d(3, 768, kernel_size=(1, 16, 16), stride=(1, 16, 16))
      )
    )
    (layer): ModuleList(
      (0-11): 12 x VJEPA21Layer(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attention): VJEPA21RopeAttention(
          (query): Linear(in_features=768, out_features=768, bias=True)
          (key): Linear(in_features=768, out_features=768, bias=True)
          (value): Linear(in_features=768, out_features=768, bias=True)
          (proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (drop_path): Identity()
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): VJEPA21MLP(
          (fc1): Linear(in_features=768, 

In [10]:
# 2. Inspect Config Architecture Features
print("=== Model Architecture Features ===")
print(f"Model Type:            {getattr(config, 'model_type', 'N/A')}")
print(f"Hidden Dimension (D):  {config.hidden_size}")
print(f"Image Resolution:      {getattr(config, 'image_size', 384)} x {getattr(config, 'image_size', 384)}")
print(f"Patch Size:            {config.patch_size} x {config.patch_size}")
print(f"Tubelet Size (Frames): {getattr(config, 'tubelet_size', 2)}")
print(f"Num Layers / Heads:    {config.num_hidden_layers} layers / {config.num_attention_heads} heads")

# Calculate spatial and temporal patch counts
spatial_patches = (getattr(config, "image_size", 384) // config.patch_size) ** 2
print(f"Patches per Frame:     (384/16)^2 = {spatial_patches}")

=== Model Architecture Features ===
Model Type:            vjepa21
Hidden Dimension (D):  768
Image Resolution:      384 x 384
Patch Size:            16 x 16
Tubelet Size (Frames): 2
Num Layers / Heads:    12 layers / 12 heads
Patches per Frame:     (384/16)^2 = 576


In [11]:
(384/16)**2

576.0

In [12]:
# 3. Test Forward Pass to Verify Input & Output Dimensions
# Expected Input Shape: (Batch, Channels, Frames, Height, Width)
batch_size = 1
channels = 3
frames = 16  # Standard clip length (must be divisible by tubelet_size=2)
height, width = 384, 384

dummy_input = torch.randn(batch_size, channels, frames, height, width)

print("\n=== Input / Output Inspection ===")
print(
    f"Required Input Shape: [Batch={batch_size}, Channels={channels}, Frames={frames}, Height={height}, Width={width}]"
)

with torch.no_grad():
    outputs = model(dummy_input)

# Check output structure
if hasattr(outputs, "last_hidden_state"):
    output_tensor = outputs.last_hidden_state
else:
    output_tensor = outputs[0]

print(f"Output Tensor Shape:  {list(output_tensor.shape)}")
print(f"  - Batch Size:        {output_tensor.shape[0]}")
print(
    f"  - Sequence Tokens:   {output_tensor.shape[1]} (Computed as: (16 frames / 2 tubelet_size) * 576 spatial patches)"
)
print(f"  - Feature Dim:       {output_tensor.shape[2]}")


=== Input / Output Inspection ===
Required Input Shape: [Batch=1, Channels=3, Frames=16, Height=384, Width=384]
Output Tensor Shape:  [1, 4608, 768]
  - Batch Size:        1
  - Sequence Tokens:   4608 (Computed as: (16 frames / 2 tubelet_size) * 576 spatial patches)
  - Feature Dim:       768


In [13]:
(16/2) * 576 

4608.0

In [14]:
(32/2) * 576 

9216.0

# Check example h5 file from testing

In [24]:
import h5py

h5_path = "data/output_features/train_for_testing_shard_02.h5"

with h5py.File(h5_path, "r") as f:
    print("Keys in h5 file:", list(f.keys()))
    
    
    print(f"\n------------------")
    
    features = f["features"]
    labels = f["labels"]
    names = f["names"]
    
    print(f"  features shape: {features.shape} | dtype: {features.dtype}")
    print(f"  labels shape:   {labels.shape}   | dtype: {labels.dtype}")
    print(f"  names shape:    {names.shape}    | dtype: {names.dtype}")
    
    # Sample inspection
    print(f"\n------------------")
    for i in [0]:
        name = names[i].decode("utf-8") if isinstance(names[i], bytes) else names[i]
        label = labels[i]
        print(f"  Sample {i} -> Name: {name}, Label: {label}")

Keys in h5 file: ['features', 'labels', 'names']

------------------
  features shape: (8, 4608, 768) | dtype: float16
  labels shape:   (8,)   | dtype: int64
  names shape:    (8,)    | dtype: object

------------------
  Sample 0 -> Name: 199780.webm, Label: 30


# Try AttentiveProbe class
Check if it works with the new input

In [16]:
import torch.nn as nn

class AttentiveProbe(nn.Module):
    """Attentive probe similar to V-JEPA, but with an additional absolute positional encoding for image models
    """
    def __init__(self, dim_embed, num_frames=16, spatial_size=16, num_heads=6, pos_enc=True, feature_key='dino_feat', num_classes=174):
        super().__init__()
        self.dim_embed = dim_embed
        self.feature_key = feature_key
        self.num_classes = num_classes
        self.query = nn.Parameter(torch.zeros(1, 1, dim_embed))
        nn.init.trunc_normal_(self.query, std=0.02)

        self.norm_attn = nn.LayerNorm(dim_embed)
        self.attn = nn.MultiheadAttention(
            embed_dim=dim_embed,
            num_heads=num_heads,
            batch_first=True,
        )

        self.norm_mlp = nn.LayerNorm(dim_embed)
        self.mlp = nn.Sequential(
            nn.Linear(dim_embed, dim_embed * 4),
            nn.GELU(),
            nn.Linear(dim_embed * 4, dim_embed),
        )

        self.linear = nn.Linear(self.dim_embed, self.num_classes)

        if pos_enc:
            pos = _get_3d_sincos_pos_embed(num_frames, spatial_size,
                                        spatial_size, dim_embed)
            self.register_buffer('pos_embed', pos.unsqueeze(0))  # [1, T*H*W, D]
        else:
            self.pos_embed = 0
    
    def forward_feat(self, data):
        assert self.feature_key in data, f'"{self.feature_key}" does not exist in the data.'

        x = data[self.feature_key]              # [B, T, D] / [B, T, HW, D]
        if x.dim() == 4:
            # x = x[:, torch.randperm(x.shape[1])]  # random temporal order
            # x = x[:,8].unsqueeze(1)               # center frame
            x = x.flatten(1,2)
        B = x.shape[0]                          # [B, T, D]
        x = x + self.pos_embed

        query = self.query.expand(B, -1, -1)    # [B, 1, D]

        # Cross-attention block (pre-norm)
        x = self.norm_attn(x)
        query = query + self.attn(query, x, x)[0]

        # MLP block (pre-norm)
        query = query + self.mlp(self.norm_mlp(query))

        return query.squeeze(1)                  # [B, D]

    def forward(self, data):
        feat = self.forward_feat(data)
        return self.linear(feat)
    

In [18]:
batch_size = 5
features = 4608
feature_dim = 768

dummy_input = torch.randn(batch_size, features, feature_dim)

model = AttentiveProbe(
    dim_embed=feature_dim, 
    num_frames=16, 
    spatial_size=16, 
    num_heads=2, 
    pos_enc=False, 
    feature_key='vjepa_feat', 
    num_classes=174
)
model.eval()


AttentiveProbe(
  (norm_attn): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (attn): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
  )
  (norm_mlp): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (mlp): Sequential(
    (0): Linear(in_features=768, out_features=3072, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=3072, out_features=768, bias=True)
  )
  (linear): Linear(in_features=768, out_features=174, bias=True)
)